# 停止条件

In [1]:
from pathlib import Path

import requests

from src.core import *
from src.gpt import *

In [2]:
np.random.seed(42)

In [3]:
class GPTTrainerWithStopping(GPTTrainer):

    def generate(self, dataset, prompt, steps=256, stop_sequences=None):
        self.model.eval()
        tokens = dataset.encode(prompt)
        prompt_len = len(tokens)
        stops = [dataset.encode(s) for s in (stop_sequences or [])]

        with Tensor.no_grad():
            for _ in range(steps):
                feature = Tensor([tokens[-dataset.context_size:]])
                logits = self.model(feature)

                last_logits = logits.data[0, -1]
                exp = np.exp(last_logits - np.max(last_logits))
                probs = exp / np.sum(exp)
                token = np.random.choice(len(probs), p=probs)
                tokens.append(token)

                generated = tokens[prompt_len:]
                hit = next((s for s in stops if self._ends_with(generated, s)), None)
                if hit is not None:
                    break

        return dataset.decode(tokens)

    @staticmethod
    def _ends_with(tokens, sequence):
        return len(tokens) >= len(sequence) and tokens[-len(sequence):] == sequence

In [4]:
BATCH_SIZE = 16
CONTEXT_SIZE = 256
EMBEDDING_SIZE = 384
HEADS = 6
BLOCKS = 6

In [5]:
DATA_FILE = "../../../tinyshakespeare.txt"

file = Path(DATA_FILE)
if not file.exists():
    file.parent.mkdir(parents=True, exist_ok=True)
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    response = requests.get(url)
    response.raise_for_status()
    file.write_text(response.text)

In [6]:
MODEL_FILE = "../../../tinyshakespeare-gpt.npz"

dataset = CharDataset(DATA_FILE, BATCH_SIZE, CONTEXT_SIZE)
model = GPT(dataset.vocab_size, CONTEXT_SIZE, EMBEDDING_SIZE, HEADS, BLOCKS)
loss_fn = GPTLoss()
optimizer = AdamWOptimizer(model.parameters)
trainer = GPTTrainerWithStopping(model, loss_fn, optimizer)
trainer.load(MODEL_FILE)

In [7]:
print(trainer.generate(dataset, prompt="ROMEO:"))

ROMEO:
By her as deserved. You avoid wides,
Sir Rome should have you to peace, and you, I.

MARCIUS:
All holy you accusident two business.
I have attorsed your right
Of direction: hereafter is novereign so.

ANGELO:
Fairl!' how he loays and home, with half as po


In [8]:
print(trainer.generate(dataset, prompt="ROMEO:", steps=64))

ROMEO:
He restile--Gentle give you without'd him;
And he denied within


In [9]:
print(trainer.generate(dataset, prompt="ROMEO:", stop_sequences=[".", "?", "!"]))

ROMEO:
Of heaven things weeping successons, go for life
France noble freerfections to savour free dullow,
And I, my lord brother, thankface thee revice:
Upon the queen you are my noble about!


In [10]:
newline_token = dataset.encode("\n")[0]
print(trainer.generate(dataset, prompt="ROMEO:", stop_sequences=["\n\n"]))

ROMEO:
My sir, next not so.


